In [1]:
import pandas as pd
import os
import numpy as np
import openpyxl
import copy
import matplotlib.pyplot as plt
import re

seed=1234
np.random.seed(seed)

In [2]:
data = pd.read_excel('./DAT/Health checkup sample.xlsx',engine='openpyxl')

In [3]:
data_copy= data.copy()

In [4]:
# labdata남기고 제거 
df = data_copy.drop(['흡연기간_과거', '하루흡연량_과거', '흡연기간_현재', '하루흡연량_현재', '음주습관_횟수', '1회음주량_보통', '1회음주량_과음','격렬운동_분', '중강도운동_분', 
              
    #labdata 중에 RH, 기생충은 대부분 하나값이라 제거, AntiHbs는 HBsAg로 구분되어 제거, VERL,TPHA는 매독관련 약음성 기준 모호하여 제거 #            
    'RH','기생충_윤충','기생충_원충','AntiHBs', 'VDRL','TPHA',

    '자궁경부암','HPV','유방촬영', '흉부촬영', '복부초음파', '위내시경', '식도생검', '위장생검', '십이지장생검', '내시경_생검_조직_HP_염색', 's자형대장경', '전대장경', '결장생검',
    'CT_뇌', 'CT_경추', 'CT_흉부', 'CT_복부', 'CT_골반', 'CT_요추', 'CT_관상동맥', 'CT_복부골반', 'CT_폐', 'CT_뇌혈관', 'CT_칼슘스코어', 
    'PETCT_Whole_Body', 'PETCT_Torso', 'PETCT_뇌', 'MRI_뇌', 'MRI_요추', 'MRI_경추', 'MRI_뇌_STROKE', 
                     'BMD_L', 'T_Score_L', 'T_Score_L_per', 'Z_Score_L', 'Z_Score_L_per', 'BMD_H', 'T_Score_H', 'T_Score_H_per', 'Z_Score_H',
       'Z_Score_H_per', '요추촬영', 'X_ray_측면', 'X_ray_무릎', 'ABI_R', 'ABI_L', 'ECG', '심장초음파', 'LVEF', 'LV_mass',
       'LV_mass_index', 'E_E_ratio', 'TRV_max', '시력나안_좌', '시력교정_좌', '시력안압_좌', '안저_좌', '시력나안_우', '시력교정_우', '시력안압_우',
       '안저_우', '좌250hz', '좌500hz', '좌1000hz', '좌2000hz', '좌3000hz', '좌4000hz', '좌6000hz', '좌8000hz', 
                     '우250hz','우500hz', '우1000hz', '우2000hz', '우3000hz', '우4000hz', '우6000hz', '우8000hz', '악력_좌', '악력_우', 'BDI'],axis=1) #86개

In [5]:
#부등호제거 함수
def replace_inequality(data,*Feature):

    for feature in Feature:
        data[feature] = data[feature].astype(str).str.replace('<','')
        data[feature] = data[feature].astype(str).str.replace('>','')
        data[feature] = data[feature].astype(str).str.replace('>=','')
    return data

In [6]:
df = replace_inequality(df,'CRP','HemoglobinA1c','C_peptide','호모시스테인',
                        'CK_MB','Troponin_T','Vit_D','TSH','FreeT4','T3','FSH',
                        'LH','E2','Testosterone','AFP','CEA','CA19_9',
                        'PSA','Helicobacter','FreeT4','요비중')

In [7]:
def map_with_regex(df, col, patterns, values, default=np.nan):
    condlist = [df[col].str.contains(p, na=False) for p in patterns]
    return np.select(condlist, values, default=default)

In [8]:
binary_vars = ["HBsAg", "Anti_HIV", "AntiHCV", "대변잠혈"]
for col in binary_vars:
    df[col] = map_with_regex(df, col,
                                    ["음성", "양성"],
                                    [0, 1],
                                    default=df[col])

In [9]:
df["HAV_Ab_lgG"] = map_with_regex(
    df, "HAV_Ab_lgG",
    ["음성", "Trace", "양성|약양성"],  # ← 여러 키워드는 | 로 묶기
    [0, 0, 1],
    default=df["HAV_Ab_lgG"]
)

In [10]:
urine_patterns = [
    "0-3/N|0-1|0-3|1-3",
    "3-5|>3-5 / T",
    ">5-10 / 1+|5-10",
    ">10-20 / 2+|10-20",
    ">20-50 / 3+|20-30|30-50",
    ">50 / 4+|50-100",
    "100이상"
]
urine_values = [0,1,2,3,4,5,6]

for col in ["요백혈구","요적혈구","요상피세포"]:
    df[col] = map_with_regex(df, col, urine_patterns, urine_values, default=df[col])

In [11]:
df['격렬운동_일수'] = df['격렬운동_일수'].replace({0.5:0, 1.5:1})

In [12]:
# 요비중을 숫자로 변환 (문자열 -> NaN)
df['요비중'] = pd.to_numeric(df['요비중'], errors='coerce')

# 범주화 조건 설정
ConditionList = [
    df['요비중'] < 1.003,
    df['요비중'] > 1.03,
    (df['요비중'].between(1.003, 1.03))
]

ChoiceList = [1, 1, 0]

df['요비중'] = np.select(ConditionList, ChoiceList, default=np.nan)

In [13]:
# Tumor Marker Test(TMT) indicator
tmt = ['AFP', 'CEA', 'CA19_9']

# threshold setting
thresholds = {"AFP": 20, "CEA": 5, "CA19_9": 37}

# 변환 (조건 충족 시 1, 아니면 0)
for col, th in thresholds.items():
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = (df[col] >= th).astype(int)

In [14]:
df['ABO'] = df['ABO'].map({'A':0,'B':1,'AB':2,'O':3,'Cis-AB':np.nan})

In [15]:
# Variables with more than 20% missing values
drop_colname =['PSA','AntiHCV','Anti_HIV','C_peptide','호모시스테인','Troponin_T','CK_MB','APOliporotein_A','APOliporotein_B','Vit_D','LDH','Mg','CA125','HAV_Ab_lgG','E2','LH',
             'FSH','Testosterone','DHEA_S','대변잠혈','PEFR','요빌리루빈','요질산염','케톤체','요당','요상피세포','BandNeut','유로빌리노겐','RA','요잠혈반응','HBsAg'] # 21 개 // 10 개
df.drop(labels=drop_colname,axis=1,inplace=True)

In [16]:
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# dfX = df.iloc[:, 3:]
# dfX = dfX.apply(pd.to_numeric, errors='coerce')
# dfX = dfX.fillna(dfX.median()) 
# vif = pd.DataFrame()
# vif["VIF Factor"] = [variance_inflation_factor(dfX.values, i) for i in range(dfX.shape[1])]
# vif["features"] = dfX.columns
# vif = vif.sort_values("VIF Factor", ascending=False).reset_index(drop=True)
# vif[vif['VIF Factor'] > 10].head(5)

In [17]:
vif_drop = ['체중','신장', 'FVC_Percent','FEV1_Percent','FEV1','mcv','mch','mchc','헤모글로빈','PolysegNeut','LDL_Cholesterol'] # 11개

df.drop(labels=vif_drop,axis=1,inplace=True)

In [18]:
print(len(pd.unique(df[df.pat_sbst_no.duplicated(keep=False)]['pat_sbst_no'])),'명')

269 명


In [19]:
# 당뇨선별기준
def create_DM(data):

    ConditionList = [
        (data['공복혈당'] >= 126) | (data['HemoglobinA1c'] >= 6.5), # 당뇨 환자
        ((data['공복혈당'] < 126) & (data['공복혈당'] >= 100)) | ((data['HemoglobinA1c'] < 6.5) & (data['HemoglobinA1c'] >= 5.7)), # 당뇨 전 단계
        (data['공복혈당'] < 100) | (data['HemoglobinA1c'] < 5.7) # 정상
    ]

    ChoiceList = [2,1,0]

    data.insert(2,'DM',np.select(ConditionList,ChoiceList,default=3))
    # 2번째 컬럼에 피실험자의 상태 변수 추가, dafault : 삼중 조건문에 해당하지 않는 경우 값 리턴

    # 건강 검진 2회 이상 받은 환자를 바탕으로 선정

    data = data[data.pat_sbst_no.duplicated(keep=False)] # duplicated 중복 데이터 확인

    stand_2 = pd.unique(data[data['DM']==3]['pat_sbst_no'].values) # example: include 'NaN'

    data = data[~data['pat_sbst_no'].isin(stand_2)] # ~ : not
    data.reset_index(inplace=True,drop=True) # index 초기화
    return data

To identify subjects with normal results at their first examination, and whose follow-up examinations showed either normal status or pre-diabetes.

In [20]:
exclude_cols = ['pat_sbst_no', 'il_inspecdate']
mask = ~df.columns.isin(exclude_cols)
df.loc[:, mask] = df.loc[:, mask].apply(pd.to_numeric, errors='coerce')

In [21]:
df = create_DM(df);df_copy=df.copy()

### Exclude participants with pre-diabetes or diabetes at the first examination.

In [22]:
df = df[~(np.isin(df['pat_sbst_no'],df[((df['순번']==1) & (df['DM']==1))]['pat_sbst_no'].values))] # 첫 검진 결과가 당뇨 전 단계인 경우 제외되고 남은 수
df = df[~(np.isin(df['pat_sbst_no'],df[((df['순번']==1) & (df['DM']==2))]['pat_sbst_no'].values))] # 첫 검진 결과가 당뇨인 경우 제외되고 남은 수

# 검진 주기

In [23]:
def logic_delete_data(data,k=1): # k : 건강검진 주기
    idx_pat = data['pat_sbst_no']
    idx_sequence = data['순번']
    idx_year = data['il_inspecdate']

    conditionList =[
            (idx_pat.eq(idx_pat.shift(-1))) &  (idx_year.shift(-1).sub(idx_year)==k),
            (idx_pat.eq(idx_pat.shift(+1))) &  (idx_year.shift(+1).sub(idx_year)== -k),
    ]

    Choicelogic = [True,True]

    data.loc[:,['logic']] = np.select(conditionList,Choicelogic,default=False) # return True / False
    data = data[data['logic']==True] # True 행만 호출
    del data['logic'];

    data_copy = data.copy()

    return data_copy

In [24]:
One_year = logic_delete_data(df)
Two_year = logic_delete_data(df,k=2) # ex) k=2 2년마다 검진 받은 피실험자 호출, k=1 1년마다 검진 받은 피실험자 호출
Two_year.reset_index(drop=True, inplace=True)
One_year.reset_index(drop=True, inplace=True)

종속 변수 생성

In [25]:
def create_label(data, k, mode='Normal'):

    data = data[data['pat_sbst_no'].isin(pd.unique(data['pat_sbst_no'])[np.where(data.groupby(by=['pat_sbst_no'])['순번'].count()==k+1)[0]])] # 특정 횟수 만큼 건강 검진 받은 환자 대상
    idx_stand=data['DM'] # 상태변수
    idx_pat=data['pat_sbst_no'] # ID
    sequence = data['순번']


    # 정상 -> 정상 : 0, 정상 -> 당뇨전단계 : 1 (Normal)
    # dataset 1
    if mode == 'Normal':
        ConditionList = [
            (idx_pat.eq(idx_pat.shift(-1))) & (sequence.shift(-1).sub(sequence) == 1) & ((idx_stand==0) & (idx_stand.shift(-1)==0)),
            (idx_pat.eq(idx_pat.shift(-1))) & (sequence.shift(-1).sub(sequence) == 1) & ((idx_stand==0) & (idx_stand.shift(-1)==1))
        ]
    # 당뇨전단계 -> 당뇨전단계 : 0, 당뇨전단계 -> 당뇨 : 1 (PreDM)
    # dataset 2
    elif mode == 'PreDM':
        ConditionList = [
        (idx_pat.eq(idx_pat.shift(-1))) & (sequence.shift(-1).sub(sequence) == 1) & ((idx_stand==1) & (idx_stand.shift(-1)==1)),
        (idx_pat.eq(idx_pat.shift(-1))) & (sequence.shift(-1).sub(sequence) == 1) & ((idx_stand==1) & (idx_stand.shift(-1)==2))
    ]

    ChoiceList = [0,1]

    data.insert(1,'label',np.select(ConditionList,ChoiceList,default=2)) # 원래 label 위치 3이었는데 1로 수정(220930)

    copy_data = data.copy()
    copy_data = copy_data[~(copy_data['label']==2)] # 이중 조건문에 해당되지 않는 경우 2 return

    # Overlap Pat_sbst_no Delete

    copy_data.drop_duplicates(subset=['pat_sbst_no','label'],keep='first',inplace=True)
    copy_data.drop_duplicates(subset=['pat_sbst_no'],keep='last',inplace=True)

    #copy_data.reset_index(inplace=True,drop=True)

    return copy_data

In [26]:
Two_year_dataset_1 = create_label(Two_year,1)
Two_year_dataset_2 = create_label(Two_year,2)
Two_year_dataset_3 = create_label(Two_year,3) # 건강 검진 4번 받은 사람

Two_year_dataset_Normal = ((pd.concat([Two_year_dataset_1,Two_year_dataset_2,Two_year_dataset_3])).sort_values(by='pat_sbst_no')).reset_index(drop=True)
##########################################################################

One_year_dataset_1 = create_label(One_year,1)
One_year_dataset_2 = create_label(One_year,2)
One_year_dataset_3 = create_label(One_year,3)
One_year_dataset_4 = create_label(One_year,4)
One_year_dataset_5 = create_label(One_year,5)
One_year_dataset_6 = create_label(One_year,6)


One_year_dataset_Normal = ((pd.concat([One_year_dataset_1,One_year_dataset_2,One_year_dataset_3,\
                                One_year_dataset_4,One_year_dataset_5,One_year_dataset_6])).sort_values(by='pat_sbst_no')).reset_index(drop=True)

# PreDM vs DM

In [27]:
df_copy = df_copy[~(np.isin(df_copy['pat_sbst_no'],df_copy[((df_copy['순번']==1) & (df_copy['DM']==0))]['pat_sbst_no'].values))] # 첫 검진 결과가 정상인 경우 제외
df_copy = df_copy[~(np.isin(df_copy['pat_sbst_no'],df_copy[((df_copy['순번']==1) & (df_copy['DM']==2))]['pat_sbst_no'].values))] # 첫 검진 결과가 당뇨인 경우 제외

df_copy.reset_index(drop=True, inplace=True)

Two_year_preDM = logic_delete_data(df_copy,2)
One_year_preDM = logic_delete_data(df_copy)

In [28]:
Two_year_dataset_11 = create_label(Two_year_preDM,1,mode='PreDM')
Two_year_dataset_21 = create_label(Two_year_preDM,2,mode='PreDM')
Two_year_dataset_31 = create_label(Two_year_preDM,3,mode='PreDM')

Two_year_dataset_PreDM = ((pd.concat([Two_year_dataset_11,Two_year_dataset_21,Two_year_dataset_31])).sort_values(by='pat_sbst_no')).reset_index(drop=True)
##########################################################################

One_year_dataset_11 = create_label(One_year_preDM,1,mode='PreDM')
One_year_dataset_21 = create_label(One_year_preDM,2,mode='PreDM')
One_year_dataset_31 = create_label(One_year_preDM,3,mode='PreDM')
One_year_dataset_41 = create_label(One_year_preDM,4,mode='PreDM')
One_year_dataset_51 = create_label(One_year_preDM,5,mode='PreDM')
One_year_dataset_61 = create_label(One_year_preDM,6,mode='PreDM')

One_year_dataset_PreDM = ((pd.concat([One_year_dataset_11,One_year_dataset_21,One_year_dataset_31,\
                                One_year_dataset_41,One_year_dataset_51,One_year_dataset_61])).sort_values(by='pat_sbst_no')).reset_index(drop=True)

# PR outlier

In [29]:
np.where(One_year_dataset_Normal['맥박수']>200)

(array([], dtype=int64),)

In [30]:
Patient = ['순번','DM','il_inspecdate','약물치료_당뇨병', '진단여부_당뇨병'] # 11개 -> 약물 치료 빼고 5개로 수정

One_year_dataset_Normal.drop(labels=Patient,axis=1,inplace=True)
One_year_dataset_PreDM.drop(labels=Patient,axis=1,inplace=True)
Two_year_dataset_Normal.drop(labels=Patient,axis=1,inplace=True)
Two_year_dataset_PreDM.drop(labels=Patient,axis=1,inplace=True)

In [31]:
def var_translate(data):
    data.rename(columns={'exam_age':'Age', 'sex':'Sex', '체지방율':'BFP', 'BMI' : 'BMI' ,'수축기혈압':'SBP', '이완기혈압':'DBP','허리둘레':'WC', 'FEV1_FVC_percent' : 'FEV1_FVC_percent', 
                         '헤마토크리트':'Hct','총콜레스테롤':'TC','요백혈구':'Urine_WBC',
                                 '음주습관':'Alcohol','가족력_당뇨병':'FH_DM','흡연상태':'Smoking','격렬운동_일수':'VIA','맥박수':'PR', '백혈구':'WBC', '적혈구':'RBC', '헤모글로빈':'Hb',
                                  'rdw' : 'RDW' ,'혈소판수':'PLT', 'Lymphocyte' : 'Lymphocyte','Monocyte' : 'Monocyte', 'Eosinophill' : 'Eosinophil', 'Basophill' : 'Basophil',
                                  'ABO' : 'Blood_type', 'ESR' : 'ESR', 'CRP':'CRP','RA':'RA','총단백':'TP','알부민':'ALB','총빌리루빈':'TBil','직접빌리루빈':'DBil','AST_SGOT':'AST','ALT_SGPT':'ALT',
                                  '중강도운동_일수':'MIA','감마지피티':'GGT','ALP':'ALP','혈액요소질소':'BUN','혈중크레아티닌':'Cr','GFR':'eGFR','요산':'Uric_Acid','중성지방':'TG',
                                  'HDL_Cholesterol':'HDL','LDL_Cholesterol':'LDL','공복혈당':'FBG','HemoglobinA1c' : 'HbA1c','총칼슘':'Ca','무기인':'P', 'Na':'Na',
                                  'K':'K', 'TSH':'TSH','FreeT4':'FreeT4', 'T3':'T3', 'AFP':'AFP','CEA':'CEA', 'CA19_9':'CA19_9',
                                  'Helicobacter':'Helicobacter','산도':'Urine_pH','요단백':'Urine_Protein','요비중':'Urine_SG','요적혈구':'Hematuria','가족력_뇌졸중_중풍':'FH_STK',
                                  '가족력_심장병_심근경색_협심증':'FH_HTDZ','가족력_고혈압':'FH_HTN','가족력_기타_암포함':'FH_CC','진단여부_뇌졸중_중풍':'PHX_STK','약물치료_뇌졸중_중풍':'DRUG_STK',
                                  '진단여부_심장병_심근경색_협심증':'PHX_HTDZ','약물치료_심장병_심근경색_협심증':'DRUG_HTDZ','진단여부_고혈압':'PHX_HTN','약물치료_고혈압':'DRUG_HTN',
                                  '진단여부_고지혈증':'PHX_HPLPDM','약물치료_고지혈증':'DRUG_HPLPDM','진단여부_폐결핵':'PHX_PHSS',
                                  '약물치료_폐결핵':'DRUG_PHSS','진단여부_기타_암포함':'PHX_CC','약물치료_기타_암포함':'DRUG_CC'},inplace=True)
    return data

In [32]:
One_year_dataset_Normal = var_translate(One_year_dataset_Normal)
Two_year_dataset_Normal =var_translate(Two_year_dataset_Normal)
One_year_dataset_PreDM=var_translate(One_year_dataset_PreDM)
Two_year_dataset_PreDM=var_translate(Two_year_dataset_PreDM)

In [33]:
# Smoking

One_year_dataset_Normal['Smoking'] = One_year_dataset_Normal['Smoking'].replace({3:0})
Two_year_dataset_Normal['Smoking'] = Two_year_dataset_Normal['Smoking'].replace({3:0})

One_year_dataset_PreDM['Smoking'] = One_year_dataset_PreDM['Smoking'].replace({3:0})
Two_year_dataset_PreDM['Smoking'] = Two_year_dataset_PreDM['Smoking'].replace({3:0})

# # Alcohol

One_year_dataset_Normal['Alcohol'] = One_year_dataset_Normal['Alcohol'].replace({4:0})
Two_year_dataset_Normal['Alcohol'] = Two_year_dataset_Normal['Alcohol'].replace({4:0})

One_year_dataset_PreDM['Alcohol'] = One_year_dataset_PreDM['Alcohol'].replace({4:0})
Two_year_dataset_PreDM['Alcohol'] = Two_year_dataset_PreDM['Alcohol'].replace({4:0})

In [34]:
writer = pd.ExcelWriter('./DAT/(XAI)Analysis_sample.xlsx')

One_year_dataset_Normal.to_excel(writer,index=False,sheet_name='1Y(0-0 vs 0-1)')
Two_year_dataset_Normal.to_excel(writer,index=False,sheet_name='2Y(0-0 vs 0-1)')
One_year_dataset_PreDM.to_excel(writer,index=False,sheet_name='1Y(1-1 vs 1-2)')
Two_year_dataset_PreDM.to_excel(writer,index=False,sheet_name='2Y(1-1 vs 1-2)')

writer.close()